In [1]:
import torch
from torch import nn
from torch.nn import functional as F

net = nn.Sequential(nn.Linear(20, 256), nn.ReLU(), nn.Linear(256, 10))

X = torch.rand(2, 20)
net(X)

tensor([[-0.2263,  0.0429, -0.2388, -0.0891,  0.1388,  0.3008, -0.0514,  0.0438,
         -0.2214,  0.0883],
        [-0.1840,  0.0551, -0.2203, -0.0799,  0.1293,  0.2063,  0.0128,  0.0477,
         -0.2059,  0.1467]], grad_fn=<AddmmBackward0>)

In [2]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(20, 256)
        self.out = nn.Linear(256, 10)

    def forward(self, X):
        return self.out(F.relu(self.hidden(X)))

In [3]:
net = MLP()
net(X)

tensor([[-0.0854,  0.1962,  0.1558,  0.1426, -0.1956, -0.0875,  0.0560,  0.0316,
         -0.1285, -0.0930],
        [-0.0646,  0.1877,  0.1562,  0.2084, -0.2387,  0.0054,  0.0587,  0.0402,
         -0.1189, -0.0473]], grad_fn=<AddmmBackward0>)

**Sequential Module**

In [5]:
class MySequential(nn.Module):
    def __init__(self, *args):
        super().__init__()  #3 super inits and inherents from the parent
        for block in args:
            self._modules[block] = block
            # every nn.Module has an internal dictionary called _modules
            # layer object itself is the name and value

    def forward(self, X):
        for block in self._modules.values():
            # loops through all layers in the internal dictionary in the exact order
            X = block(X)
        return X

net = MySequential(nn.Linear(20, 256), nn.ReLU(), nn.Linear(256, 10))
net(X)

tensor([[-0.0815,  0.0614, -0.1044,  0.0958,  0.1057, -0.2410,  0.2417, -0.0744,
          0.0683,  0.2787],
        [ 0.0146,  0.0911, -0.1370,  0.0683,  0.1264, -0.2512,  0.1575, -0.1423,
          0.0952,  0.2943]], grad_fn=<AddmmBackward0>)

**self defined class is less constraining compared to when using sequential**

In [6]:
class FixedHiddenMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.rand_weight = torch.rand((20, 20), requires_grad=False)
        self.linear = nn.Linear(20, 20)

    def forward(self, X):
        X = self.linear(X)
        X = F.relu(torch.mm(X, self.rand_weight) + 1)  # custom matrix multiplication for weights and then relu
        X = self.linear(X)

        while X.abs().sum() > 1:
            X /= 2
        return X.sum()

net = FixedHiddenMLP()
net(X)

tensor(0.0350, grad_fn=<SumBackward0>)

**Mix and matching modules**

In [8]:
class NestMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(20, 64), nn.ReLU(), 
                                 nn.Linear(64, 32), nn.ReLU())
        self.linear = nn.Linear(32, 16)

    def forward(self, X):
        return self.linear(self.net(X))

chimera = nn.Sequential(NestMLP(), nn.Linear(16, 20), FixedHiddenMLP())
chimera(X)    

tensor(0.1189, grad_fn=<SumBackward0>)